# Definitions, LLM initialization

In [ ]:
import json
import os
from langchain_ollama import ChatOllama
from typing import List, Optional
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, PromptTemplate
from pydantic import BaseModel, Field

LLM_MODEL_NAME="jaro-jm/llm:latest"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

text = """Donald Trump od początku traktował konflikt w Ukrainie jako nieswoją wojnę. Nie miał cierpliwości ani zrozumienia dla ukraińskich racji. Błędnie zakładał też, że zdoła oderwać Rosję od sojuszu z Chinami. Tymczasem oba kraje mocno zacieśniły współpracę gospodarczą, a Chiny dostarczają dziś Rosjanom maszyny i technologie, na które monopol miał kiedyś Zachód. Moskwa i Pekin pamiętają czasy, gdy były podzielone. Putin i Xi są zgodni, że rozpad Związku Radzieckiego nie był koniecznością, tylko skutkiem błędów Kremla. Rosję i Chiny dzieli przepaść kulturalna, bo Rosjanie nie chcą być Chińczykami, ale to akurat ułatwia relacje. Współpraca z Europą grozi w oczach Putina zarażeniem rosyjskiego społeczeństwa „demokratycznym wirusem”, a w kontaktach z Chinami takiego ryzyka nie ma. Rosja może jednak wpaść w nieodwracalną zależność od Chin. Decydując się na inwazję na Ukrainę, Putin chciał przede wszystkim zasilić ludność Rosji dziesiątkami milionów Słowian, co było dla niego ważniejsze od zdobyczy terytorialnych. W tle są obawy przed chińską dominacją."""

# Node extraction

In [ ]:
nodes_extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """You analyze text for the purpose of building a knowledge graph.
Your task is to extract ONLY 12-17 most relevant entities, events, concepts, that can function as nodes in the knowledge graph.

Specifically identify:
- persons (individual human actors),
- organizations (companies, institutions, movements),
- locations (geographical and administrative places),
- events (historical, social, political, technical, scientific, etc.),
- time expressions (dates, periods, time ranges),
- key thematic concepts (ideas, technologies, phenomena, theories).

**Rules:**
- Return only semantically meaningful elements, that can exist independently as nodes in a knowledge graph.
- Use canonical names where possible (e.g. “European Union” instead of abbreviations, if context allows).
Avoid returning multiple surface forms of the same entity or concept.
If different names refer to the same real-world entity (e.g. „ChRL”, „Chiny”, „Chińska Republika Ludowa”), return only one canonical form, chosen based on common usage in context.
- Do not duplicate items.
- Do not add category labels, keys, explanations or metadata.
- Return everything in Polish.

**Output format:**
- A single line of text.
- All extracted elements separated by commas.
- No JSON, no lists, no bullet points, no headings, no comments."""),
    ("user", """Text:
    Donald Trump nie ogranicza się do przebudowy Białego Domu. Zmienia też to, jak Amerykanie patrzą
na świat. Wenezuela liczy się dla niego bardziej niż Ukraina, bo to z Ameryki Łacińskiej
pochodzą największe kłopoty dzisiejszych Stanów: nielegalna imigracja i przemyt narkotyków.
Ważny jest również Pacyfik i rywalizacja z Chinami o hegemonię w tym regionie.
Trump uważa, że USA traciły na porządku, który powstał po zakończeniu zimnej wojny.
Wciąż trwa spór o to, czy przyjęcie ChRL do WTO było dobrą decyzją.
Jednocześnie Ameryka wiele zyskiwała, między innymi dzięki temu,
że dolar pozostawał walutą rezerwową świata."""),
    ("ai", """Donald Trump, Biały Dom, Stany Zjednoczone, Amerykanie, Wenezuela, Ukraina, Ameryka Łacińska,
nielegalna imigracja, przemyt narkotyków, Pacyfik, Chiny, hegemonia, zimna wojna,
Światowa Organizacja Handlu, dolar, waluta rezerwowa świata"""),
    ("user", "Text: {text}")
])

nodes_extraction_chain = nodes_extraction_prompt | ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=10000, num_predict=5000)
nodes_extracted_content_raw = nodes_extraction_chain.invoke({"text": text})
nodes_list = [node.strip() for node in nodes_extracted_content_raw.content.split(",")]
print(nodes_list)

In [ ]:
nodes = ", ".join(nodes_list)
print(nodes)

# Q&A pair generation

In [ ]:
qa_generator_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert assistant for generating high-quality question–answer pairs from text using provided entities.

Rules:
- Generate 6-8 question–answer pairs in Polish.
- Use ONLY entities listed below to form questions:\n{nodes}
- Do NOT ask generic or meta-level questions about the text itself. Forbidden example of a question: "Jakie jest główne zagadnienie omawiane w tekście?"
- Questions must test understanding of the content, not summarize it at a high level.
- Avoid trivial, vague, or purely definitional questions. Questions and answers MUST cover ONLY the most important aspects of the text.
- Answers must be concise, precise, and based only on the provided text.
- Resolve ambiguities by clearly attributing statements, claims, or actions to specific people or entities, indicating who thinks, says, or does what, rather than leaving information vague or general.
- Use canonical names where possible (e.g. “European Union” instead of abbreviations, if context allows).
Avoid returning multiple surface forms of the same entity or concept.
If different names refer to the same real-world entity (e.g. „ChRL”, „Chiny”, „Chińska Republika Ludowa”), return only one canonical form, chosen based on common usage in context.

Output format:
- Return ONLY valid JSON.
- The JSON must be a list of objects.
- Each object must have exactly two fields: "question" and "answer".
- Do NOT include markdown, comments, or any text outside the JSON.

Example output:
[
  {{
    "question": "Jaką rolę pełni mechanizm X w opisywanym procesie?",
    "answer": "Mechanizm X odpowiada za ..."
  }},
  {{
    "question": "Jakie konsekwencje wynikają z zastosowania rozwiązania Y?",
    "answer": "Zastosowanie rozwiązania Y prowadzi do ..."
  }}
]"""),
    ("user", """Generate question–answer pairs from the following text:\n{text}""")
])

llm_chain = qa_generator_prompt | ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=10000, num_predict=5000)
output = llm_chain.invoke({"text": text, "nodes": nodes})
qa_pairs = json.loads(output.content)
all_pairs = [f"Pytanie {i+1}: {pair['question']}\nOdpowiedź {i+1}: {pair['answer']}" for i, pair in enumerate(qa_pairs)]
qa_single_sting = "\n\n".join(all_pairs)
print(qa_single_sting)

# Relations pipeline

## Prompts, helper classes

In [ ]:
class UnstructuredRelation(BaseModel):
    head: str = Field(
        description=(
            "extracted head entity like Microsoft, Apple, John. "
            "Must use human-readable unique identifier."
        )
    )
    head_type: str = Field(
        description="type of the extracted head entity like Person, Company, etc"
    )
    relation: str = Field(description="relation between the head and the tail entities")
    tail: str = Field(
        description=(
            "extracted tail entity like Microsoft, Apple, John. "
            "Must use human-readable unique identifier."
        )
    )
    tail_type: str = Field(
        description="type of the extracted tail entity like Person, Company, etc"
    )

examples = [
    {
        "question-answer pairs": "Pytanie 1: Jak Trump ocenił przebieg rozmowy z Putinem?\nOdpowiedź 1: 'Jako dobrą i bardzo produktywną'.",
        "head": "rozmowa z Władimirem Putinem",
        "head_type": "Wydarzenie",
        "relation": "MA_OCENĘ",
        "tail": "dobra i bardzo produktywna",
        "tail_type": "Opinia",
    },
    {
        "question-answer pairs": "Pytanie 1: Kiedy zaplanowane było spotkanie Trumpa z prezydentem Ukrainy Zełenskim?\nOdpowiedź 1: Dziś o godzinie 13:00 (19:00 czasu polskiego).",
        "head": "spotkanie z Zełenskim",
        "head_type": "Wydarzenie",
        "relation": "ZAPLANOWANE_NA",
        "tail": "13:00 (19:00 czasu polskiego)",
        "tail_type": "Czas",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Kto przekazał informację o rozmowie telefonicznej z prezydentem Rosji?\n"
            "Odpowiedź 1: Donald Trump.\n\n"
            "Pytanie 2: Na jakiej platformie Donald Trump opublikował tę informację?\n"
            "Odpowiedź 2: Truth Social."
        ),
        "head": "Donald Trump",
        "head_type": "Osoba",
        "relation": "PRZEKAZAŁ_INFORMACJĘ",
        "tail": "rozmowa z Władimirem Putinem",
        "tail_type": "Wydarzenie",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Kto przekazał informację o rozmowie telefonicznej z prezydentem Rosji?\n"
            "Odpowiedź 1: Donald Trump.\n\n"
            "Pytanie 2: Na jakiej platformie Donald Trump opublikował tę informację?\n"
            "Odpowiedź 2: Truth Social."
        ),
        "head": "Donald Trump",
        "head_type": "Osoba",
        "relation": "UŻYŁ_PLATFORMY",
        "tail": "Truth Social",
        "tail_type": "Platforma",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Jakie przykłady sankcji wskazał polityk?\n"
            "Odpowiedź 1: Sankcje ze strony USA i Europy, w tym dotyczące sektora naftowego i finansowego, które mają znacząco utrudniać funkcjonowanie irańskiej gospodarki."
        ),
        "head": "Sankcje USA i Europy",
        "head_type": "Wydarzenie",
        "relation": "UTRUDNIA",
        "tail": "funkcjonowanie irańskiej gospodarki",
        "tail_type": "Wydarzenie",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Jaki przykład działań prezydenta Iranu wskazał?\n"
            "Odpowiedź 1: Uderzenie przez Iran w amerykańską bazę wojskową w Katarze w czerwcu br., w odpowiedzi na atak USA na irańskie obiekty nuklearne z użyciem bomb GBU-57."
        ),
        "head": "Atak na irańskie obiekty nuklearne",
        "head_type": "Wydarzenie",
        "relation": "WYWOŁAŁ",
        "tail": "uderzenie Iranu w bazę USA w Katarze",
        "tail_type": "Wydarzenie",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Jaki przykład działań prezydenta Iranu wskazał?\n"
            "Odpowiedź 1: Uderzenie przez Iran w amerykańską bazę wojskową w Katarze w czerwcu br., w odpowiedzi na atak USA na irańskie obiekty nuklearne z użyciem bomb GBU-57."
        ),
        "head": "Stany Zjednoczone",
        "head_type": "Państwo",
        "relation": "ZAATAKOWAŁY",
        "tail": "irańskie obiekty nuklearne",
        "tail_type": "Obiekt",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Jaki przykład działań prezydenta Iranu wskazał?\n"
            "Odpowiedź 1: Uderzenie przez Iran w amerykańską bazę wojskową w Katarze w czerwcu br., w odpowiedzi na atak USA na irańskie obiekty nuklearne z użyciem bomb GBU-57."
        ),
        "head": "Iran",
        "head_type": "Państwo",
        "relation": "ZAATAKOWAŁY",
        "tail": "amerykańska baza wojskowa w Katarze",  # A specific base in Qatar - the Americans have many bases, so in this case we do not want a link to the 'Katar' node
        "tail_type": "Obiekt",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Jaki przykład działań prezydenta Iranu wskazał?\n"
            "Odpowiedź 1: Uderzenie przez Iran w amerykańską bazę wojskową w Katarze w czerwcu br., w odpowiedzi na atak USA na irańskie obiekty nuklearne z użyciem bomb GBU-57."
        ),
        "head": "Stany Zjednoczone",
        "head_type": "Państwo",
        "relation": "UŻYŁY",
        "tail": "bomby GBU-57",
        "tail_type": "Uzbrojenie",
    }
]

examples_decomposition = [
    {
        "question-answer pairs": (
            "Pytanie 1: Jakie były działania Putina oraz jego motywacje związane z inwazją na Ukrainę?\n"
            "Odpowiedź 1: Putin zainicjował inwazję na Ukrainę, obawiał się rosnącej dominacji Chin i chciał wzmocnić demograficznie Rosję."
        ),
        "head": "Vladimir Putin",
        "head_type": "Osoba",
        "relation": "ZAINICJOWAŁ",
        "tail": "Inwazja na Ukrainę",
        "tail_type": "Wydarzenie",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Jakie były działania Putina oraz jego motywacje związane z inwazją na Ukrainę?\n"
            "Odpowiedź 1: Putin zainicjował inwazję na Ukrainę, obawiał się rosnącej dominacji Chin i chciał wzmocnić demograficznie Rosję."
        ),
        "head": "Vladimir Putin",
        "head_type": "Osoba",
        "relation": "OBAWIAŁ_SIĘ",
        "tail": "Chińska dominacja",
        "tail_type": "Koncepcja",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Jakie były działania Putina oraz jego motywacje związane z inwazją na Ukrainę?\n"
            "Odpowiedź 1: Putin zainicjował inwazję na Ukrainę, obawiał się rosnącej dominacji Chin i chciał wzmocnić demograficznie Rosję."
        ),
        "head": "Inwazja na Ukrainę",
        "head_type": "Wydarzenie",
        "relation": "MIAŁA_NA_CELU",
        "tail": "wzmocnienie demograficzne",
        "tail_type": "Koncepcja",
    },
    {
        "question-answer pairs": (
            "Pytanie 1: Jakie były działania Putina oraz jego motywacje związane z inwazją na Ukrainę?\n"
            "Odpowiedź 1: Putin zainicjował inwazję na Ukrainę, obawiał się rosnącej dominacji Chin i chciał wzmocnić demograficznie Rosję."
        ),
        "head": "wzmocnienie demograficzne",
        "head_type": "Koncepcja",
        "relation": "DOTYCZY",
        "tail": "Rosja",
        "tail_type": "Państwo",
    }
]

## Helper methods

In [ ]:
def create_prompt(nodes: Optional[str] = None, rels: Optional[str] = None) -> ChatPromptTemplate:
    base_string_parts = [
        f"""You are a top-tier algorithm designed for extracting information in structured formats to build a knowledge graph. Your task is to identify relations from a given question-answer pairs requested with the user prompt.

        You must generate the output in a JSON format containing a list with JSON objects. Each object should have the keys: "head", "head_type", "relation", "tail", and "tail_type".
        The "head" key must contain the text of the entity with one of the entities from the provided list in the user prompt.
        The "head_type" key must contain the type of the head entity.
        The "relation" key must contain the type of relationship between the "head" and the "tail.
        The "tail" key must represent the text of an entity which is the tail of the relation and the "tail_type" key must contain the type of the tail entity. The "tail" key must be one of the entities from the provided list in the user prompt.

        Maintain entity consistency:
        When using entities, it's vital to ensure consistency. If an entity, such as "John Doe", is mentioned multiple times in the question-answer pairs but is referred to by different names or pronouns (e.g., "Joe", "he"), always use the identifier from provided list of entities for that entity.
        The knowledge graph should be coherent and easily understandable, so maintaining consistency in entity references is crucial.

        IMPORTANT NOTES:
        - Don't add any explanation and text.
        - Return a list of JSON objects with all values in Polish."""
    ]
    system_message = SystemMessage(content="\n".join(filter(None, base_string_parts)))
    parser = JsonOutputParser(pydantic_object=UnstructuredRelation)

    human_string_parts = [
        """Based on the following examples extract 8-10 relations that carry clear, non-trivial informational value from the provided question-answer pairs.
        If no meaningful relation can be formed between two entities, do not create one. When a statement in the text expresses a complex or multi-part fact, decompose it into multiple simple relations.
        Each relation must represent a single, atomic factual claim. Do NOT compress multiple facts into a single vague relation. Prefer multiple clear relations over one overloaded relation.""",
        f"""Use the following entities, don't use other entity that is not defined below:\n# ENTITIES:\n{nodes}""" if nodes else "",
        f"""Use the following relations, avoid using other relation that is not defined below:\n# RELATIONS:\n{rels}""" if rels else "",
        """Below are a number of examples of question-answer pairs and their extracted entities and relationships.

        Examples - simple extraction:
        {examples}

        Below are a number of examples of correct decomposition (complex relations):
        {examples_decomposition}

        For the following question-answer pairs, extract entities and relations as in the provided example.
        {format_instructions}

        Question-answer pairs: {input}"""
    ]
    human_prompt_string = "\n".join(filter(None, human_string_parts))
    human_message_prompt = HumanMessagePromptTemplate(prompt=PromptTemplate(template=human_prompt_string, input_variables=["input"], partial_variables={"format_instructions": parser.get_format_instructions(), "nodes": nodes, "rels": rels, "examples": examples, "examples_decomposition": examples_decomposition}))
    chat_prompt = ChatPromptTemplate.from_messages([system_message, human_message_prompt])
    return chat_prompt

## Relations generation

In [ ]:
import time

start = time.time()
rels_generator_prompt = create_prompt(nodes)

collected_chunks = []

llm_chain = rels_generator_prompt | ChatOllama(
    model=LLM_MODEL_NAME,
    temperature=0,
    reasoning=False,
    num_ctx=10000,
    num_predict=125000
)

print("Initialization time:", time.time() - start, "seconds")

for chunk in llm_chain.stream({"input": qa_single_sting}):
    if hasattr(chunk, "content"):
        text = chunk.content
    else:
        text = str(chunk)
    print(text, end="", flush=True)
    collected_chunks.append(text)

triples_raw_text = "".join(collected_chunks)
triples = json.loads(triples_raw_text)

for t in triples:
    print(f"{t['head']} ({t['head_type']}) "
          f"──[{t['relation']}]──▶ "
          f"{t['tail']} ({t['tail_type']})")

# By summary

In [ ]:
summary_generator_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert assistant for generating high-quality summaries of provided text.

Rules:
Return ONLY a valid summary string in Polish, do NOT include markdown, comments, or any text beside the summary.
Resolve ambiguities by clearly attributing statements, claims, or actions to specific people or entities, indicating who thinks, says, or does what, rather than leaving information vague or general.
The summary must be factual, neutral, and concise.
Your summary should consist of 3 to 4 sentences only.
Do NOT add interpretations, opinions, speculation, or external knowledge.
Preserve causal relationships, actors, and concrete actions.
Remove rhetoric, narrative framing, and redundancy.
Prefer explicit facts over vague generalizations.
Preserve exact numbers, dates, and named entities when present.
Focus on what happened and who did what; include claims only if they are themselves materially relevant events."""),
    ("user", """Generate summary of the following text:\n{text}""")
])

llm_chain = summary_generator_prompt | ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=10000, num_predict=25000)
summary = llm_chain.invoke({"text": text}).content
print(summary)

## Prompts, helper classes, helper methods

In [ ]:
class UnstructuredRelation(BaseModel):
    head: str = Field(
        description=(
            "extracted head entity like Microsoft, Apple, John. "
            "Must use human-readable unique identifier."
        )
    )
    head_type: str = Field(
        description="type of the extracted head entity like Person, Company, etc"
    )
    relation: str = Field(description="relation between the head and the tail entities")
    tail: str = Field(
        description=(
            "extracted tail entity like Microsoft, Apple, John. "
            "Must use human-readable unique identifier."
        )
    )
    tail_type: str = Field(
        description="type of the extracted tail entity like Person, Company, etc"
    )

examples = [
    {
        "text": "Trump ocenił przebieg rozmowy z Putinem jako dobrą i bardzo produktywną.",
        "head": "rozmowa z Władimirem Putinem",
        "head_type": "Wydarzenie",
        "relation": "MA_OCENĘ",
        "tail": "dobra i bardzo produktywna",
        "tail_type": "Opinia",
    },
    {
        "text": "Spotkanie Trumpa z prezydentem Ukrainy Zełenskim zaplanowano dziś o godzinie 13:00 (19:00 czasu polskiego).",
        "head": "spotkanie z Zełenskim",
        "head_type": "Wydarzenie",
        "relation": "ZAPLANOWANE_NA",
        "tail": "13:00 (19:00 czasu polskiego)",
        "tail_type": "Czas",
    },
    {
        "text": "Informację o rozmowie telefonicznej z prezydentem Rosji przekazał Donald Trump na platformie Truth Social.",
        "head": "Donald Trump",
        "head_type": "Osoba",
        "relation": "PRZEKAZAŁ_INFORMACJĘ",
        "tail": "rozmowa z Władimirem Putinem",
        "tail_type": "Wydarzenie",
    },
    {
        "text": "Informację o rozmowie telefonicznej z prezydentem Rosji przekazał Donald Trump na platformie Truth Social.",
        "head": "Donald Trump",
        "head_type": "Osoba",
        "relation": "UŻYŁ_PLATFORMY",
        "tail": "Truth Social",
        "tail_type": "Platforma",
    },
    {
        "text": "Uderzenie przez Iran w amerykańską bazę wojskową w Katarze w czerwcu br., w odpowiedzi na atak USA na irańskie obiekty nuklearne z użyciem bomb GBU-57.",
        "head": "Atak na irańskie obiekty nuklearne",
        "head_type": "Wydarzenie",
        "relation": "WYWOŁAŁ",
        "tail": "uderzenie Iranu w bazę USA w Katarze",
        "tail_type": "Wydarzenie",
    },
    {
        "text": "Uderzenie przez Iran w amerykańską bazę wojskową w Katarze w czerwcu br., w odpowiedzi na atak USA na irańskie obiekty nuklearne z użyciem bomb GBU-57.",
        "head": "Stany Zjednoczone",
        "head_type": "Państwo",
        "relation": "ZAATAKOWAŁY",
        "tail": "irańskie obiekty nuklearne",
        "tail_type": "Obiekt",
    },
    {
        "text": "Uderzenie przez Iran w amerykańską bazę wojskową w Katarze w czerwcu br., w odpowiedzi na atak USA na irańskie obiekty nuklearne z użyciem bomb GBU-57.",
        "head": "Iran",
        "head_type": "Państwo",
        "relation": "ZAATAKOWAŁY",
        "tail": "amerykańska baza wojskowa w Katarze",  # A specific base in Qatar - the Americans have many bases, so in this case we do not want a link to the 'Katar' node
        "tail_type": "Obiekt",
    },
    {
        "text": "Uderzenie przez Iran w amerykańską bazę wojskową w Katarze w czerwcu br., w odpowiedzi na atak USA na irańskie obiekty nuklearne z użyciem bomb GBU-57.",
        "head": "Stany Zjednoczone",
        "head_type": "Państwo",
        "relation": "UŻYŁY",
        "tail": "bomby GBU-57",
        "tail_type": "Uzbrojenie",
    }
]

examples_decomposition = [
    {
        "text": "Putin zainicjował inwazję na Ukrainę, obawiał się rosnącej dominacji Chin i chciał wzmocnić demograficznie Rosję.",
        "head": "Vladimir Putin",
        "head_type": "Osoba",
        "relation": "ZAINICJOWAŁ",
        "tail": "Inwazja na Ukrainę",
        "tail_type": "Wydarzenie",
    },
    {
        "text": "Putin zainicjował inwazję na Ukrainę, obawiał się rosnącej dominacji Chin i chciał wzmocnić demograficznie Rosję.",
        "head": "Vladimir Putin",
        "head_type": "Osoba",
        "relation": "OBAWIAŁ_SIĘ",
        "tail": "Chińska dominacja",
        "tail_type": "Koncepcja",
    },
    {
        "text": "Putin zainicjował inwazję na Ukrainę, obawiał się rosnącej dominacji Chin i chciał wzmocnić demograficznie Rosję.",
        "head": "Inwazja na Ukrainę",
        "head_type": "Wydarzenie",
        "relation": "MIAŁA_NA_CELU",
        "tail": "wzmocnienie demograficzne",
        "tail_type": "Koncepcja",
    },
    {
        "text": "Putin zainicjował inwazję na Ukrainę, obawiał się rosnącej dominacji Chin i chciał wzmocnić demograficznie Rosję.",
        "head": "wzmocnienie demograficzne",
        "head_type": "Koncepcja",
        "relation": "DOTYCZY",
        "tail": "Rosja",
        "tail_type": "Państwo",
    }
]

def create_prompt(nodes: Optional[str] = None, rels: Optional[str] = None) -> ChatPromptTemplate:
    base_string_parts = [
        f"""You are a top-tier algorithm designed for extracting information in structured formats to build a knowledge graph. Your task is to identify relations from a given text requested with the user prompt.

        You must generate the output in a JSON format containing a list with JSON objects. Each object should have the keys: "head", "head_type", "relation", "tail", and "tail_type".
        The "head" key must contain the text of the entity with one of the entities from the provided list in the user prompt.
        The "head_type" key must contain the type of the head entity.
        The "relation" key must contain the type of relationship between the "head" and the "tail.
        The "tail" key must represent the text of an entity which is the tail of the relation and the "tail_type" key must contain the type of the tail entity. The "tail" key must be one of the entities from the provided list in the user prompt.

        Maintain entity consistency:
        When using entities, it's vital to ensure consistency. If an entity, such as "John Doe", is mentioned multiple times in the text but is referred to by different names or pronouns (e.g., "Joe", "he"), always use the identifier from provided list of entities for that entity.
        The knowledge graph should be coherent and easily understandable, so maintaining consistency in entity references is crucial.

        IMPORTANT NOTES:
        - Don't add any explanation and text.
        - Return a list of JSON objects with all values in Polish."""
    ]
    system_message = SystemMessage(content="\n".join(filter(None, base_string_parts)))
    parser = JsonOutputParser(pydantic_object=UnstructuredRelation)

    human_string_parts = [
        """Based on the following examples extract 6-7 relationships that carry clear, non-trivial informational value from the provided text.
        If no meaningful relation can be formed between two entities, do not create one. When a statement in the text expresses a complex or multi-part fact, decompose it into multiple simple relations.
        Each relation must represent a single, atomic factual claim. Do NOT compress multiple facts into a single vague relation. Prefer multiple clear relations over one overloaded relation.""",
        f"""Use the following entities, don't use other entity that is not defined below:\n# ENTITIES:\n{nodes}""" if nodes else "",
        f"""Use the following relations, avoid using other relation that is not defined below:\n# RELATIONS:\n{rels}""" if rels else "",
        """Below are a number of examples of texts and their relationships.

        Examples - simple extraction:
        {examples}

        Below are a number of examples of correct decomposition (complex relations):
        {examples_decomposition}

        For the following text, extract entities and relations as in the provided example.
        {format_instructions}

        Text: {input}"""
    ]
    human_prompt_string = "\n".join(filter(None, human_string_parts))
    human_message_prompt = HumanMessagePromptTemplate(prompt=PromptTemplate(template=human_prompt_string, input_variables=["input"], partial_variables={"format_instructions": parser.get_format_instructions(), "nodes": nodes, "rels": rels, "examples": examples, "examples_decomposition": examples_decomposition}))
    chat_prompt = ChatPromptTemplate.from_messages([system_message, human_message_prompt])
    return chat_prompt

In [ ]:
rels_generator_prompt = create_prompt(nodes)
llm_chain = rels_generator_prompt | ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=25000, num_predict=25000)

collected_chunks = []
for chunk in llm_chain.stream({"input": qa_single_sting}):
    if hasattr(chunk, "content"):
        text = chunk.content
    else:
        text = str(chunk)
    print(text, end="", flush=True)
    collected_chunks.append(text)

triples_raw_text = "".join(collected_chunks)
triples = json.loads(triples_raw_text)

for t in triples:
    print(f"{t['head']} ({t['head_type']}) "
          f"──[{t['relation']}]──▶ "
          f"{t['tail']} ({t['tail_type']})")

In [ ]:
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(
    llm=ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=25000, num_predict=25000),
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""Rules:
- Output language: Polish.
- Extract 20-25 entities and 10-15 relationships between them from provided text (fewer if the text does not support more without loss of quality).
- Relationship keys must be specific and informative, not vague or generic. Try to split relations between multiple JSON objects if context of a text requires it.
- Use canonical names where possible (e.g. "European Union" instead of abbreviations, if context allows).
- Do not create meta-nodes (e.g., Autor, Tekst, Artykuł). First-person statements should be reduced to the underlying objective fact connecting real entities."""
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(summary)])
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")